In [14]:
import pandas as pd
pd.set_option('display.max_colwidth', 100)  

In [15]:
df = pd.read_csv('/Users/ujjwalbhatta/Desktop/ecommerce-product-success-predictor/data/raw/electronics_merged_30k.csv')
df.head()

,parent_asin,main_category,sub_category,product_title,description,price,average_rating,rating_number,brand,store,details,rating,review_title,text,helpful_vote,verified_purchase,review_date
0,B00DSX557O,Computers,Electronics,KDLINKS® Super Speed USB 3.0 to 10/100/1000M Gigabit Ethernet LAN Network Adapter - 2015 New Mod...,"['New Model with Latest Technology', 'Add Gigabit Ethernet network connectivity to a Laptop or D...",12.95,3.1,56,KDLINKS,KDLINKS,"{'Package Dimensions': '6.8 x 4.2 x 2.4 inches', 'Item Weight': '1 Pounds', 'Manufacturer': 'KDL...",5.0,Works.........,What's not to like. Plugged in and it worked.,NaN,True,NaN
1,B001OTZ8DA,All Electronics,Electronics,Sennheiser HD 800 Reference Dynamic Headphone,['Sennheiser HD 800 Reference Dynamic Headphone.'],NaN,4.0,232,Sennheiser Consumer Audio,Sennheiser Consumer Audio,"{'Brand': 'Sennheiser Consumer Audio', 'Model Name': 'Language _ tag', 'Color': 'Grey', 'Form Fa...",5.0,"If you got the money and want better quality audio, these will do the trick.",Nice Quality. Definitely an improvement over what I have been using. $1350+ worth? Meh.... ...,NaN,True,NaN
2,B00SQGAV7C,Computers,Electronics,MSI GT Series GT80 Titan SLI-071 18.4-Inch Laptop (Aluminum Black),"['Product Description', 'MSI GT80 Titan SLI-071: 18.4"" Full HD Wide View Anti-glare Cutting-edge...",NaN,3.1,14,MSI,MSI,"{'Standing screen display size': '18.4 Inches', 'Max Screen Resolution': '1920 x 1080 Pixels', '...",4.0,CHECK THE Warranty & BEware of Seller Adorama CAMERA,The MSI COMPUTER GT80 TITAN has been adquately reviewed by others. This review is to warn other...,NaN,False,NaN
3,B0878VWPZM,Cell Phones & Accessories,Electronics,"Firstick Remote Cover Glow in The Dark,Silicone Case for FirTVStick4K/FirTV Stick (2nd Gen)/ Fir...",[],8.99,4.7,4942,Auswaur,Auswaur,"{'Product Dimensions': '6.18 x 3.23 x 0.59 inches', 'Item Weight': '0.739 ounces', 'Best Sellers...",2.0,More For Looks Than Protection [Edited],When was the last time you broke a remote? I can't remember going back 30-40 years. Maybe one? S...,NaN,False,NaN
4,B0063R728M,All Electronics,Electronics,Acer K330 Portable Home Theater Projector,"['Product Description', ""The Acer K330 Projector delivers HD 720p video entertainment with a nat...",NaN,3.0,26,Acer America Corporation,acer,"{'Product Dimensions': '8.6 x 6.6 x 1.8 inches', 'Item Weight': '2.73 pounds', 'Item model numbe...",3.0,Decent projector,"Prior to purchasing this projector, I had never seen an LED project but was impressed with the i...",NaN,False,NaN


In [16]:
# Find the number of null values in each column
null_counts = df.isnull().sum()

# Print the columns with null values and their respective counts
print("Columns with null values and their counts:")
print(null_counts[null_counts > 0])

Columns with null values and their counts:
main_category      503
sub_category      2613
product_title        3
price            18374
store              126
review_title         4
text                 2
helpful_vote     30000
review_date      30000
dtype: int64


In [17]:
# Drop exact duplicates
df = df.drop_duplicates(subset=['parent_asin', 'review_title', 'text'])

# Drop columns that are all null
df = df.drop(columns=['helpful_vote', 'review_date'])

# Handle null brand
df['brand'] = df['brand'].fillna('UnknownBrand')

# Handle null store
df['store'] = df['store'].fillna('UnknownStore')

# Handle null text
df['review_title'] = df['review_title'].fillna('No Title')

# Handle null review title
df['text'] = df['text'].fillna('No Text')

# Handle null main category
df['main_category'] = df['main_category'].fillna('Unknown')

# Handle null sub category
df['sub_category'] = df['sub_category'].fillna('Unknown')

# Handle null sub category !!!! we may need to drop these price if its important factor we can check at last
df['price'] = pd.to_numeric(df['price'], errors='coerce')    # convert to float; invalids become NaN
df['price'] = df.groupby('main_category')['price'].transform(lambda x: x.fillna(x.median()))
# Drop rows where price is still null after imputation
df = df.dropna(subset=['price'])

print(f"Rows after dropping null prices: {len(df)}")

df.head()


Rows after dropping null prices: 29970


,parent_asin,main_category,sub_category,product_title,description,price,average_rating,rating_number,brand,store,details,rating,review_title,text,verified_purchase
0,B00DSX557O,Computers,Electronics,KDLINKS® Super Speed USB 3.0 to 10/100/1000M Gigabit Ethernet LAN Network Adapter - 2015 New Mod...,"['New Model with Latest Technology', 'Add Gigabit Ethernet network connectivity to a Laptop or D...",12.95,3.1,56,KDLINKS,KDLINKS,"{'Package Dimensions': '6.8 x 4.2 x 2.4 inches', 'Item Weight': '1 Pounds', 'Manufacturer': 'KDL...",5.0,Works.........,What's not to like. Plugged in and it worked.,True
1,B001OTZ8DA,All Electronics,Electronics,Sennheiser HD 800 Reference Dynamic Headphone,['Sennheiser HD 800 Reference Dynamic Headphone.'],25.99,4.0,232,Sennheiser Consumer Audio,Sennheiser Consumer Audio,"{'Brand': 'Sennheiser Consumer Audio', 'Model Name': 'Language _ tag', 'Color': 'Grey', 'Form Fa...",5.0,"If you got the money and want better quality audio, these will do the trick.",Nice Quality. Definitely an improvement over what I have been using. $1350+ worth? Meh.... ...,True
2,B00SQGAV7C,Computers,Electronics,MSI GT Series GT80 Titan SLI-071 18.4-Inch Laptop (Aluminum Black),"['Product Description', 'MSI GT80 Titan SLI-071: 18.4"" Full HD Wide View Anti-glare Cutting-edge...",28.96,3.1,14,MSI,MSI,"{'Standing screen display size': '18.4 Inches', 'Max Screen Resolution': '1920 x 1080 Pixels', '...",4.0,CHECK THE Warranty & BEware of Seller Adorama CAMERA,The MSI COMPUTER GT80 TITAN has been adquately reviewed by others. This review is to warn other...,False
3,B0878VWPZM,Cell Phones & Accessories,Electronics,"Firstick Remote Cover Glow in The Dark,Silicone Case for FirTVStick4K/FirTV Stick (2nd Gen)/ Fir...",[],8.99,4.7,4942,Auswaur,Auswaur,"{'Product Dimensions': '6.18 x 3.23 x 0.59 inches', 'Item Weight': '0.739 ounces', 'Best Sellers...",2.0,More For Looks Than Protection [Edited],When was the last time you broke a remote? I can't remember going back 30-40 years. Maybe one? S...,False
4,B0063R728M,All Electronics,Electronics,Acer K330 Portable Home Theater Projector,"['Product Description', ""The Acer K330 Projector delivers HD 720p video entertainment with a nat...",25.99,3.0,26,Acer America Corporation,acer,"{'Product Dimensions': '8.6 x 6.6 x 1.8 inches', 'Item Weight': '2.73 pounds', 'Item model numbe...",3.0,Decent projector,"Prior to purchasing this projector, I had never seen an LED project but was impressed with the i...",False


In [18]:
# Find the number of null values in each column
null_counts = df.isnull().sum()

# Print the columns with null values and their respective counts
print("Columns with null values and their counts:")
print(null_counts[null_counts > 0])



Columns with null values and their counts:
product_title    3
dtype: int64


In [19]:
# Find the number of null values in each column
null_counts = df.isnull().sum()

# Print the columns with null values and their respective counts
print("Columns with null values and their counts:")
print(null_counts[null_counts > 0])



Columns with null values and their counts:
product_title    3
dtype: int64


In [20]:
# Unique categories
categories = df['main_category'].unique()
print(categories)

category_counts = df['main_category'].value_counts()
print(category_counts)

subcategory_counts = df['sub_category'].value_counts()
print(subcategory_counts)

brand_counts = df['brand'].value_counts()
print(brand_counts)

unique_brand_counts = df['brand'].unique()
print(brand_counts)


['Computers' 'All Electronics' 'Cell Phones & Accessories'
 'Camera & Photo' 'Office Products' 'Amazon Home' 'Musical Instruments'
 'Tools & Home Improvement' 'Unknown' 'Home Audio & Theater'
 'Car Electronics' 'Amazon Devices' 'Industrial & Scientific'
 'AMAZON FASHION' 'Apple Products' 'Toys & Games' 'Health & Personal Care'
 'Sports & Outdoors' 'Grocery' 'Automotive' 'Portable Audio & Accessories'
 'Arts, Crafts & Sewing' 'GPS & Navigation' 'Amazon Fire TV' 'Video Games'
 'All Beauty' 'Baby' 'Software' 'Books']
main_category
All Electronics                 8543
Computers                       7490
Camera & Photo                  3211
Home Audio & Theater            2893
Cell Phones & Accessories       2872
Amazon Devices                  1304
Industrial & Scientific          526
Unknown                          503
Tools & Home Improvement         443
Office Products                  371
Amazon Home                      347
Car Electronics                  313
AMAZON FASHION        

In [21]:
# almost similar cat and subcat so drop subcat
df = df.drop(columns=['sub_category'])

# clean category text
def clean_category(cat):
    return str(cat).strip().title()

df['main_category'] = df['main_category'].apply(clean_category)

category_counts = df['main_category'].value_counts()
small_categories = category_counts[category_counts < 150].index
df['main_category'] = df['main_category'].replace(small_categories, 'Other')

category_counts = df['main_category'].value_counts()
category_counts


main_category
All Electronics              8543
Computers                    7490
Camera & Photo               3211
Home Audio & Theater         2893
Cell Phones & Accessories    2872
Amazon Devices               1304
Other                         768
Industrial & Scientific       526
Unknown                       503
Tools & Home Improvement      443
Office Products               371
Amazon Home                   347
Car Electronics               313
Amazon Fashion                198
Sports & Outdoors             188
Name: count, dtype: int64

In [22]:
# Count brand frequencies
brand_counts = df['brand'].value_counts()
print(f"Original unique brands: {df['brand'].nunique()}")
print(f"Brands with <20 products: {(brand_counts < 20).sum()}")

# Brands with <20 products into 'OTHER_BRAND'
rare_brands = brand_counts[brand_counts < 20].index
df['brand_consolidated'] = df['brand'].replace(rare_brands, 'Other')

print(f"Products moved to other: {(df['brand_consolidated'] == 'Other').sum()}")

Original unique brands: 10648
Brands with <20 products: 10490
Products moved to other: 20095


In [23]:
import re

def clean_review_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)           # URLs
    text = re.sub(r'<[^>]+>', '', text)           # HTML tags
    text = re.sub(r'br\s+', ' ', text)            # "br" tags
    text = re.sub(r'[^\w\s\.\!\?\,\-]', ' ', text)  
    text = re.sub(r'\s+', ' ', text)              # spaces
    return text.strip()

def clean_product_text(text):
    if pd.isna(text):
        return text
    text = str(text).lower()
    text = re.sub(r'[^\w\s\-]', ' ', text)        # only letters, numbers, spaces, hyphens
    text = re.sub(r'\s+', ' ', text)              # spaces
    return text.strip()

df['text'] = df['text'].apply(clean_review_text)
df['review_title'] = df['review_title'].apply(clean_review_text) 
df['description'] = df['description'].apply(clean_review_text)
df['product_title'] = df['product_title'].apply(clean_product_text)
df['description'] = df['description'].replace('', 'no description')
df['brand'] = df['brand'].str.title().str.strip()
df['store'] = df['store'].str.title().str.strip()

In [24]:
df['price'] = df['price'].round(2)
df['verified_purchase'] = df['verified_purchase'].astype(bool)
df['rating'] = df['rating'].astype(int)
df.head()

,parent_asin,main_category,product_title,description,price,average_rating,rating_number,brand,store,details,rating,review_title,text,verified_purchase,brand_consolidated
0,B00DSX557O,Computers,kdlinks super speed usb 3 0 to 10 100 1000m gigabit ethernet lan network adapter - 2015 new mode...,"new model with latest technology , add gigabit ethernet network connectivity to a laptop or desk...",12.95,3.1,56,Kdlinks,Kdlinks,"{'Package Dimensions': '6.8 x 4.2 x 2.4 inches', 'Item Weight': '1 Pounds', 'Manufacturer': 'KDL...",5,works.........,what s not to like. plugged in and it worked.,True,Other
1,B001OTZ8DA,All Electronics,sennheiser hd 800 reference dynamic headphone,sennheiser hd 800 reference dynamic headphone.,25.99,4.0,232,Sennheiser Consumer Audio,Sennheiser Consumer Audio,"{'Brand': 'Sennheiser Consumer Audio', 'Model Name': 'Language _ tag', 'Color': 'Grey', 'Form Fa...",5,"if you got the money and want better quality audio, these will do the trick.",nice quality. definitely an improvement over what i have been using. 1350 worth? meh.... i wish ...,True,Sennheiser Consumer Audio
2,B00SQGAV7C,Computers,msi gt series gt80 titan sli-071 18 4-inch laptop aluminum black,"product description , msi gt80 titan sli-071 18.4 full hd wide view anti-glare cutting-edge gami...",28.96,3.1,14,Msi,Msi,"{'Standing screen display size': '18.4 Inches', 'Max Screen Resolution': '1920 x 1080 Pixels', '...",4,check the warranty beware of seller adorama camera,the msi computer gt80 titan has been adquately reviewed by others. this review is to warn other ...,False,MSI
3,B0878VWPZM,Cell Phones & Accessories,firstick remote cover glow in the dark silicone case for firtvstick4k firtv stick 2nd gen firsti...,no description,8.99,4.7,4942,Auswaur,Auswaur,"{'Product Dimensions': '6.18 x 3.23 x 0.59 inches', 'Item Weight': '0.739 ounces', 'Best Sellers...",2,more for looks than protection edited,when was the last time you broke a remote? i can t remember going back 30-40 years. maybe one? s...,False,Other
4,B0063R728M,All Electronics,acer k330 portable home theater projector,"product description , the acer k330 projector delivers hd 720p video entertainment with a native...",25.99,3.0,26,Acer America Corporation,Acer,"{'Product Dimensions': '8.6 x 6.6 x 1.8 inches', 'Item Weight': '2.73 pounds', 'Item model numbe...",3,decent projector,"prior to purchasing this projector, i had never seen an led project but was impressed with the i...",False,Other


In [25]:
import os
os.makedirs("../data/processed", exist_ok=True)

df.to_csv("../data/processed/amazon_electronics_clean.csv", index=False)